# 📚 Notebook 1: Knowledge Base Builder
### Agriculture RAG System — Pakistan Farmers Pipeline

---
**What this notebook does:**
1. Loads your Excel file (`RAG KNOWLEDGE BASE.xlsx`)
2. Validates and cleans all columns
3. Generates vector embeddings using `SentenceTransformer`
4. Stores everything into a **persistent** ChromaDB database on disk
5. Shows a coverage summary of what's in your knowledge base

**Run this notebook ONCE** (or whenever you update your Excel file).  
Notebook 2 will load the saved database — no need to rebuild every time.

---
**Excel file required columns:**
| Column | Description |
|--------|-------------|
| `QUESTION` | The farmer's question (in English) |
| `ANSWER` | The expert answer |
| `crop` | e.g. Wheat, Paddy (Rice), Cotton, Maize |
| `topic` | e.g. Fertilizer, Pest Management, Irrigation |
| `stage` | Growth stage e.g. Vegetative, Flowering |
| `intent_type` | e.g. Recommendation, Dosage, Symptoms |
| `entity` | Specific pest/disease/fertilizer name |
| `region` | e.g. Punjab, Sindh |
| `season` | Kharif, Rabi, etc. |

## Step 0 — Install Dependencies

In [15]:
# %pip install -q pandas openpyxl sentence-transformers chromadb

## Step 1 — Configuration
Set your file paths here. Everything else is automatic.

In [16]:
# ============================================================
#  CONFIGURATION — Edit these paths as needed
# ============================================================

# Path to your Excel knowledge base
EXCEL_PATH = "./RAG_knowledgeBase.xlsx"

# Where to save the persistent ChromaDB (a folder will be created)
# Notebook 2 will read from this same path
CHROMA_DB_PATH = "./content/agriculture_chroma_db"

# Embedding model — lightweight, fast, works offline
# ~90MB download, no GPU required
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ChromaDB collection name (must match Notebook 2)
COLLECTION_NAME = "agriculture_kb"

print("Configuration loaded.")
print(f"  Excel path  : {EXCEL_PATH}")
print(f"  ChromaDB    : {CHROMA_DB_PATH}")
print(f"  Embeddings  : {EMBEDDING_MODEL}")

Configuration loaded.
  Excel path  : ./RAG_knowledgeBase.xlsx
  ChromaDB    : ./content/agriculture_chroma_db
  Embeddings  : all-MiniLM-L6-v2


## Step 2 — Import Libraries

In [17]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from collections import Counter
import os
import shutil

print("✅ All libraries imported.")

✅ All libraries imported.


## Step 3 — Load & Validate Excel File

In [18]:
REQUIRED_COLUMNS = [
    'QUESTION', 'ANSWER', 'crop', 'topic', 'stage',
    'intent_type', 'entity', 'region', 'season'
]

VALID_TOPICS = [
    'Sowing', 'Seed Rate', 'Nursery', 'Transplanting', 'Fertilizer',
    'Irrigation', 'Weed Management', 'Pest Management', 'Disease Management',
    'Harvesting', 'Yield', 'Variety', 'Soil', 'Climate',
    'Land Preparation', 'General'
]

VALID_STAGES = [
    'Pre-Sowing', 'Nursery', 'Germination', 'Vegetative', 'Flowering',
    'Fruit Formation', 'Grain Formation', 'Maturity', 'Harvest',
    'Post-Harvest', 'Any'
]

VALID_INTENT_TYPES = [
    'Recommendation', 'Prevention', 'Chemical Control', 'Biological Control',
    'Cultural Control', 'Mechanical Control', 'Symptoms', 'Impact',
    'Duration', 'Identification', 'Dosage', 'Timing', 'Fact', 'Comparison'
]

VALID_SEASONS = ['Kharif', 'Rabi', 'Spring', 'Summer', 'Winter', 'Not specified']


def load_and_validate_excel(file_path: str) -> pd.DataFrame:
    print("=" * 60)
    print("LOADING KNOWLEDGE BASE")
    print("=" * 60)

    # --- Load ---
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    print(f"\n📂 Loaded file: {file_path}")
    print(f"   Raw rows    : {len(df)}")
    print(f"   Raw columns : {list(df.columns)}")

    # --- Add missing columns ---
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        print(f"\n⚠️  Missing columns (will be auto-filled): {missing}")
        for col in missing:
            df[col] = 'Not specified'
    else:
        print("\n✅ All required columns present.")

    # --- Clean ---
    df = df.fillna('Not specified')
    df = df[df['QUESTION'].notna() & (df['QUESTION'].str.strip() != '')]
    df = df[df['ANSWER'].notna() & (df['ANSWER'].str.strip() != '')]
    df = df.reset_index(drop=True)

    # --- Strip whitespace from string columns ---
    for col in REQUIRED_COLUMNS:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()

    print(f"\n✅ Clean rows ready for indexing: {len(df)}")

    # --- Validation warnings ---
    print("\n--- Validation Warnings ---")
    invalid_topics = df[~df['topic'].isin(VALID_TOPICS + ['Not specified'])]['topic'].unique()
    if len(invalid_topics):
        print(f"⚠️  Unknown topics (will still be indexed): {list(invalid_topics)}")

    invalid_seasons = df[~df['season'].isin(VALID_SEASONS)]['season'].unique()
    if len(invalid_seasons):
        print(f"⚠️  Unknown seasons: {list(invalid_seasons)}")

    if not len(invalid_topics) and not len(invalid_seasons):
        print("✅ All metadata values look valid.")

    return df


df = load_and_validate_excel(EXCEL_PATH)
df.head(3)

LOADING KNOWLEDGE BASE

📂 Loaded file: ./RAG_knowledgeBase.xlsx
   Raw rows    : 3112
   Raw columns : ['NO', 'QUESTION', 'ANSWER', 'crop', 'topic', 'stage', 'intent_type', 'entity', 'region', 'season']

✅ All required columns present.

✅ Clean rows ready for indexing: 3112

--- Validation Warnings ---
⚠️  Unknown topics (will still be indexed): ['Flowering', 'Vegetative', 'Area', 'Production', 'Zone Planning', 'IPM', 'Seed Quality', 'Agronomy', 'Tillage', 'Seed Treatment', 'Growth Stage', 'Nutrition', 'Micronutrient', 'Physiology', 'Season', 'Fruiting', 'Statistic', 'Disease', 'Disease prevention', 'Disease control', 'Weed management', 'Insect pest', 'Insect control', 'Weed control', 'Harvest', 'Post-harvest', 'General info', 'Soil requirement', 'Climate requirement', 'Climate stress', 'Land preparation', 'Field management', 'Storage', 'Pruning', 'Orchard Management', 'Region Suitability', 'Field Preparation', 'Field Management', 'Growth & Development', 'Variety & Yield', 'Seed Select

,NO,QUESTION,ANSWER,crop,topic,stage,intent_type,entity,region,season
0,1,"For paddy (rice), what is the nursery (seedlin...","For paddy (rice), the nursery/seedling stage f...",Paddy (Rice),Nursery,Nursery,Statistic,Not specified,Pakistan-General,Not specified
1,2,"For paddy (rice) short-duration varieties, wha...","For paddy (rice) short-duration varieties, the...",Paddy (Rice),General,Vegetative,Statistic,Variety: Short-duration,Pakistan-General,Not specified
2,3,"For paddy (rice) long-duration varieties, what...","For paddy (rice) long-duration varieties, the ...",Paddy (Rice),General,Vegetative,Statistic,Variety: Long-duration,Pakistan-General,Not specified


## Step 4 — Coverage Analysis
See what crops, topics, and regions your knowledge base covers.

In [19]:
def print_coverage(df: pd.DataFrame):
    print("=" * 60)
    print("KNOWLEDGE BASE COVERAGE ANALYSIS")
    print("=" * 60)
    print(f"\nTotal Q&A pairs: {len(df)}")

    sections = [
        ('Crop', 'crop'),
        ('Topic', 'topic'),
        ('Growth Stage', 'stage'),
        ('Intent Type', 'intent_type'),
        ('Season', 'season'),
        ('Region', 'region'),
    ]

    for label, col in sections:
        print(f"\n📊 {label} Distribution:")
        counts = Counter(df[col])
        for val, cnt in counts.most_common(10):
            bar = '█' * int(cnt / len(df) * 40)
            print(f"   {val:<30} {cnt:>4} ({cnt/len(df)*100:.1f}%) {bar}")


print_coverage(df)

KNOWLEDGE BASE COVERAGE ANALYSIS

Total Q&A pairs: 3112

📊 Crop Distribution:
   Paddy (Rice)                    298 (9.6%) ███
   Cotton                          243 (7.8%) ███
   Wheat                           162 (5.2%) ██
   guava                           155 (5.0%) █
   sugarcane                       154 (4.9%) █
   Sugarcane                       122 (3.9%) █
   Maize                           118 (3.8%) █
   Garlic                          110 (3.5%) █
   General                         110 (3.5%) █
   Watermelon                       90 (2.9%) █

📊 Topic Distribution:
   Fertilizer                      345 (11.1%) ████
   Pest Management                 310 (10.0%) ███
   Disease Management              297 (9.5%) ███
   Sowing                          256 (8.2%) ███
   Irrigation                      188 (6.0%) ██
   Weed Management                 144 (4.6%) █
   Harvesting                      124 (4.0%) █
   Climate                         122 (3.9%) █
   General        

## Step 5 — Build Vector Embeddings & Save to ChromaDB
This is the main step. It converts every Q&A row into a vector and stores it persistently.
Takes a few minutes depending on how large your Excel file is.

In [24]:
def build_and_save_knowledge_base(df, chroma_path, embedding_model_name, collection_name, overwrite=True):
    print("=" * 60)
    print("BUILDING VECTOR INDEX")
    print("=" * 60)

    # --- Load embedding model ---
    print(f"\n📦 Loading embedding model: {embedding_model_name}")
    model = SentenceTransformer(embedding_model_name)
    print("✅ Embedding model ready.")

    # --- Init persistent ChromaDB ---
    print(f"\n💾 Initialising persistent ChromaDB at: {chroma_path}")
    client = chromadb.PersistentClient(path=chroma_path)
    try:
        client.delete_collection(name=collection_name)
        print(f"   Deleted existing collection '{collection_name}'")
    except Exception:
        pass
    collection = client.create_collection(name=collection_name, metadata={"hnsw:space": "cosine"})
    print(f"   Created collection '{collection_name}'")

    # --- Build documents ---
    print(f"\n🔨 Preparing {len(df)} documents...")
    documents, metadatas, ids = [], [], []
    for idx, row in df.iterrows():
        doc_text = (
            f"Question: {row['QUESTION']}\n"
            f"Answer: {row['ANSWER']}\n"
            f"Crop: {row['crop']}\n"
            f"Topic: {row['topic']}\n"
            f"Stage: {row['stage']}\n"
            f"Intent: {row['intent_type']}\n"
            f"Entity: {row['entity']}"
        )
        documents.append(doc_text)
        metadatas.append({
            'question'   : str(row['QUESTION']),
            'answer'     : str(row['ANSWER']),
            'crop'       : str(row['crop']),
            'topic'      : str(row['topic']),
            'stage'      : str(row['stage']),
            'intent_type': str(row['intent_type']),
            'entity'     : str(row['entity']),
            'region'     : str(row['region']),
            'season'     : str(row['season']),
        })
        ids.append(f"doc_{idx}")

    # --- Generate embeddings ---
    print(f"\n⚙️  Generating embeddings (batch size 64)...")
    BATCH_SIZE = 64
    all_embeddings = []
    for i in range(0, len(documents), BATCH_SIZE):
        batch = documents[i:i + BATCH_SIZE]
        batch_embeddings = model.encode(batch, show_progress_bar=False)
        all_embeddings.extend(batch_embeddings.tolist())
        print(f"   Embedded {min(i + BATCH_SIZE, len(documents))}/{len(documents)} documents")
    print(f"✅ All {len(documents)} embeddings generated.")

    # --- Insert into ChromaDB ---
    print("\n💾 Inserting into ChromaDB...")
    INSERT_BATCH = 512
    for i in range(0, len(documents), INSERT_BATCH):
        collection.add(
            documents  = documents[i:i + INSERT_BATCH],
            embeddings = all_embeddings[i:i + INSERT_BATCH],
            metadatas  = metadatas[i:i + INSERT_BATCH],
            ids        = ids[i:i + INSERT_BATCH]
        )
        print(f"   Inserted {min(i + INSERT_BATCH, len(documents))}/{len(documents)} documents")

    # --- Verify ---
    count = collection.count()
    print(f"\n✅ ChromaDB now contains {count} indexed documents.")
    print(f"📁 Saved to disk at: {chroma_path}")
    print("\n👉 You can now run Notebook 2.")
    return collection


collection = build_and_save_knowledge_base(
    df=df,
    chroma_path=CHROMA_DB_PATH,
    embedding_model_name=EMBEDDING_MODEL,
    collection_name=COLLECTION_NAME,
    overwrite=True
)


BUILDING VECTOR INDEX

📦 Loading embedding model: all-MiniLM-L6-v2
✅ Embedding model ready.

💾 Initialising persistent ChromaDB at: ./content/agriculture_chroma_db
   Deleted existing collection 'agriculture_kb'
   Created collection 'agriculture_kb'

🔨 Preparing 3112 documents...

⚙️  Generating embeddings (batch size 64)...
   Embedded 64/3112 documents
   Embedded 128/3112 documents
   Embedded 192/3112 documents
   Embedded 256/3112 documents
   Embedded 320/3112 documents
   Embedded 384/3112 documents
   Embedded 448/3112 documents
   Embedded 512/3112 documents
   Embedded 576/3112 documents
   Embedded 640/3112 documents
   Embedded 704/3112 documents
   Embedded 768/3112 documents
   Embedded 832/3112 documents
   Embedded 896/3112 documents
   Embedded 960/3112 documents
   Embedded 1024/3112 documents
   Embedded 1088/3112 documents
   Embedded 1152/3112 documents
   Embedded 1216/3112 documents
   Embedded 1280/3112 documents
   Embedded 1344/3112 documents
   Embedded 1408

## Step 6 — Quick Sanity Check
Run a test query directly against the saved ChromaDB to confirm it works.

In [25]:
from sentence_transformers import SentenceTransformer as STModel

def sanity_check(chroma_path: str, collection_name: str, embedding_model_name: str):
    print("=" * 60)
    print("SANITY CHECK — Testing saved ChromaDB")
    print("=" * 60)

    # Reload from disk
    client   = chromadb.PersistentClient(path=chroma_path)
    coll     = client.get_collection(name=collection_name)
    model    = STModel(embedding_model_name)

    print(f"\n✅ Loaded collection '{collection_name}' with {coll.count()} documents.")

    test_queries = [
        "gehuun mein zang ki bimari"        ,  # Roman Urdu: wheat rust disease
        "chawal mein kide maar dawa"         ,  # Roman Urdu: pesticide for rice
        "cotton irrigation during flowering" ,  # English
        "when to apply urea to wheat"        ,  # English
    ]

    for q in test_queries:
        emb     = model.encode(q).tolist()
        results = coll.query(query_embeddings=[emb], n_results=2)
        print(f"\n🔍 Query : \"{q}\"")
        for i, meta in enumerate(results['metadatas'][0]):
            sim = 1 - results['distances'][0][i]
            print(f"   Result {i+1} (sim={sim:.3f}): {meta['question'][:70]}")
            print(f"              Crop={meta['crop']}, Topic={meta['topic']}")

    print("\n" + "=" * 60)
    print("✅ Sanity check passed. Knowledge base is ready!")
    print("=" * 60)


sanity_check(CHROMA_DB_PATH, COLLECTION_NAME, EMBEDDING_MODEL)

SANITY CHECK — Testing saved ChromaDB

✅ Loaded collection 'agriculture_kb' with 3112 documents.

🔍 Query : "gehuun mein zang ki bimari"
   Result 1 (sim=0.366): Gandum gho ki halat mein sitta nikal rhi hai tou isko 8 din ke waqfe s
              Crop=Wheat, Topic=General
   Result 2 (sim=0.360): kitchen garden lagi hui hai ismein meine bhindhi ki fasal lagai hai au
              Crop=Chili, Topic=Pest Management

🔍 Query : "chawal mein kide maar dawa"
   Result 1 (sim=0.389): Gandum gho ki halat mein sitta nikal rhi hai tou isko 8 din ke waqfe s
              Crop=Wheat, Topic=General
   Result 2 (sim=0.382): kitchen garden lagi hui hai ismein meine bhindhi ki fasal lagai hai au
              Crop=Chili, Topic=Pest Management

🔍 Query : "cotton irrigation during flowering"
   Result 1 (sim=0.671): Why is irrigation critical during the flowering and grain-filling stag
              Crop=Millet, Topic=Irrigation
   Result 2 (sim=0.647): When should the first irrigation be applied in lin

---
## ✅ Done!

Your knowledge base is built and saved at:
```
/content/agriculture_chroma_db
```

**Next step:** Open **Notebook 2** (`2_RAG_Pipeline.ipynb`) to run the full voice-to-answer pipeline.

> ℹ️ Only re-run this notebook if you update your Excel file.